In [11]:

from __future__ import division         # for Python 2 compatibility
from __future__ import print_function   # for Python 2 compatibility
from pathlib import Path                # treat paths as objects with methods instead of strings

import matplotlib.pyplot as plt         # import of matplotlib
import numpy as np                      # import of numpy
import pandas as pd                     # data manipulation library
import glob                             # search folders and files on the computer
import os                               # built-in library for system
import openpyxl                         # read excel data
%matplotlib inline
# renders images as PNG inside VSC

# !
# ?
# *
# TODO
# // 

Last update: <br> - 25/08/14 (process combined export files), <br> - 24/11/25 (added preprocessing step), <br> - 24/11/19 (checked) <br> <br>
Goal:  To transfer generated data in PTR-Viewer (csv,) to excel as a results file 
- preprocess all csv files (remove unwanted columns)
- import data from csv file
- adjust header: remove "m" and everything after " " 
- remove all rows that contain non-numeric values (NaN, Inf) (and negative values)
- choose column AI1_Act (YM)
- find column when value 2.9 is first reached
- select all following 420 rows (this is done after filtering out the NaN values!)
- copy the header to a new Excel sheet 
- copy the rows to a new Excel sheet
- reorder sheets and remove unnecessary columns in the overview

In [ ]:
folder = r"C:\yourfolderhere"         #* Define the folder containing the CSV files - for all following activities

# Loop through all files in the folder
for filename in os.listdir(folder):
    if filename.lower().endswith('.csv'):
        file_path = os.path.join(folder, filename)                  #* This is the path of the original csv file
        df_preprocessing = pd.read_csv(file_path)                   #* Load CSV file into new DataFrame

        if len(df_preprocessing.columns) <670:                      #* Check if the file has enough columns
            print(f"Unwanted rows in file {filename} were already removed. Skipping..")
            continue
    
        # Remove all columns of raw and corrected values except of H2O clusters
        conc_columns = [col for col in df_preprocessing.columns if "(Conc)" in col]
        H2O_cluster_columns  =  [ "AbsTime", "RelTime", "Cycle", "AI1_Act", "PTR-Reaction.E/N_Act [Td]",
                                "m21.022 (H3(18)O+) (Raw)", 
                                "m21.022 (H3(18)O+) (Corr)", 
                                "m37.028 ((H3(18)O+)*H2O) (Raw)", 
                                "m37.028 ((H3(18)O+)*H2O) (Corr)", 
                                "m39.033 ((H3(18)O+)*H2(18)O) (Raw)", 
                                "m39.033 ((H3(18)O+)*H2(18)O) (Corr)", 
                                "m55.039 ((H3(18)O+)*2 H2O) (Raw)", 
                                "m55.039 ((H3(18)O+)*2 H2O) (Corr)" ]
        
        columns_to_keep = H2O_cluster_columns + conc_columns

        # If multiple files are exported into one .csv, keep the column with the filename as well
        if "Filename" in df_preprocessing.columns:
            columns_to_keep.append("Filename")
        
        filtered_df = df_preprocessing[columns_to_keep]
        filtered_df.to_csv(file_path, index = False)
        print(f"Unwanted rows in {filename} removed!")

        # For combined export files, create separate csv files depending on the filename
        if "Filename" in filtered_df.columns:
            for fname_value, group_df in filtered_df.groupby("Filename"):
                group_df = group_df.drop(columns=["Filename"])
                export_path = os.path.join(folder, f"{fname_value}.csv")
                group_df.to_csv(export_path, index=False)
                print(f"File created {export_path}")
        else:
            print(f"'Filename' column not found in {filename}, skipping...")

print("All CSV files updated!")

Unwanted rows in combined.csv removed!
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_06_39_19.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_07_00_59.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_07_15_30.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_07_30_17.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_07_45_16.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_08_00_16.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_08_15_10.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_08_29_57.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_08_45_11.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Data_09_00_20.h5.csv
File created C:\Users\install\Desktop\Ionicon\Viewer Projects

In [13]:

result_path = os.path.join(folder, 'Results.xlsx')                        #* Define the path for the new excel result file - for all following activities

# Create an ExcelWriter object
with pd.ExcelWriter(result_path, engine='openpyxl') as writer:
    sheet_added = False                                                   #* Flag to track if at least one sheet is added

    # Loop through all files in the folder
    for filename in os.listdir(folder):
        if filename.lower().endswith('.csv'):

            if os.path.splitext(filename)[0].lower().startswith("combined"):       #* Skipping combined files
                print(f"Skipping file: {filename}")
                continue        

            file_path = os.path.join(folder, filename)                    #* This is the path of the original csv file
            file = pd.read_csv(file_path)                                 #* Load CSV file into DataFrame

            if len(file.columns) <669:                                    #* Check if the file has enough columns (correct peak library used)
                print(f"File {filename} does not have enough columns. Possible error during data export. Skipping...")
                continue

            # Adjust header
            file.columns = [col if idx < 13 else col.split('(')[0].strip() for idx, col in enumerate(file.columns)]     #* Remove everything after '(' for all header columns >12
            file.columns = [col if idx < 2 else col.replace('m', '') for idx, col in enumerate(file.columns)]           #* Remove 'm' for all header columns >2

            # Remove all rows (if present) that contain non-numeric values (NaN, Inf) or a negative value
            errors = r'^[A-Za-z]+$'                                                     #* Define wrong values to sort out (all letters, upper- and lowercase)
            file = file[~file['18.010'].astype(str).str.contains(errors, na=False)]     #* Filter out rows where column '18.010' contains only letters
            file = file[file['18.010'] >= 0]                                            #* Remove rows containing negative values

            original_row_count = len(pd.read_csv(file_path))
            filtered_row_count = len(file)
            if original_row_count == filtered_row_count:
                print("Dataset ok!")
            else:
                print(f"Rows removed: {original_row_count - filtered_row_count}")

            # Filter column 'AI1_Act' for values >= 2.9 and copy the following 420 rows and the header to a new sheet
            if 'AI1_Act' in file.columns:
                voltage_selection = file[file['AI1_Act'] >= 2.9].index       #* Filter column 'AI1_Act' for values >= 2.9
                if not voltage_selection.empty:
                    start = voltage_selection[0]                             #* Find the first value where the selection is met
                    end = min(start + 420, len(file))                        #* Select the following 420 rows
                    selected_rows = file.iloc[start:end]

                    #Reorder columns so that E/N and AI1_Act values appear in front
                    cols = list(selected_rows.columns)
                    if 'AI1_Act' in cols:
                        cols.insert(0, cols.pop(cols.index('AI1_Act')))
                    if 'PTR-Reaction.E/N_Act [Td]' in cols:
                        cols.insert(0, cols.pop(cols.index('PTR-Reaction.E/N_Act [Td]')))
                    selected_rows = selected_rows[cols]

                    # Extract the sheet name from the CSV file name
                    sheet_name = filename.split('.', 1)[0]          #* Remove suffix
                    sheet_name = sheet_name[len("Data_"):]          #* Remove prefix        

                    # Add the selected rows to the results Excel sheet
                    selected_rows.to_excel(writer, index=False, sheet_name=sheet_name)
                    sheet_added = True
                    print(f"Data saved to {result_path} with sheet name '{sheet_name}'")
                else:
                    print(f"Column 'AI1_Act' has not reached desired value (2.9) in {filename}.")
            else:
                print(f"Column 'AI1_Act' not found in {filename}.")

    if not sheet_added:
        print("No sheets were added to the Excel file. Please check the input CSV files and criteria.")

print("All CSV files have been processed and saved into the Excel file.")

Skipping file: combined.csv
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '06_39_19'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '07_00_59'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '07_15_30'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '07_30_17'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '07_45_16'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '08_00_16'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '08_15_10'
Dataset ok!
Data saved to C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx with sheet name '08_29_

In [14]:
# Add average values to each sheet of the result file
result_sheets = pd.read_excel(result_path, sheet_name=None)                #* sheet_name=None loads all sheets into a new data frame

with pd.ExcelWriter(result_path, engine='openpyxl') as writer:             #* Create output file
    for sheet_name, df in result_sheets.items():                           #* Loop through the dictionary of sheets
        if len(df) >= 420:
            average = df.mean(axis=0, numeric_only=True)                   #* Calculate the mean of each column (axis=0), header is ignored
            average = pd.DataFrame(average).T                              #* Convert mean values into data frame and transpose them

            # Create new data frame where the average values are added as a new row, with one blank row in between
            average_rows = pd.concat([df.iloc[:422], pd.DataFrame(index=[None]), average, df.iloc[422:]], ignore_index=True) 
            print(f"Averaged values added to {sheet_name}")
        else:
            print(f"Not enough rows (420) in {sheet_name}. Skipping...")

        average_rows.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Averaged values have been added to each sheet in {result_path}")

Averaged values added to 06_39_19
Averaged values added to 07_00_59
Averaged values added to 07_15_30
Averaged values added to 07_30_17
Averaged values added to 07_45_16
Averaged values added to 08_00_16
Averaged values added to 08_15_10
Averaged values added to 08_29_57
Averaged values added to 08_45_11
Averaged values added to 09_00_20
Averaged values added to 09_15_23
Averaged values added to 09_30_11
Averaged values added to 09_44_59
Averaged values added to 10_00_05
Averaged values added to 10_14_51
Averaged values added to 10_30_08
Averaged values added to 10_44_52
Averaged values added to 10_59_40
Averaged values added to 11_14_47
Averaged values added to 11_29_45
Averaged values added to 11_44_49
Averaged values added to 11_59_37
Averaged values added to 12_14_18
Averaged values have been added to each sheet in C:\Users\install\Desktop\Ionicon\Viewer Projects\250815\Results.xlsx


In [15]:
# Create an overview page with all averaged values as well as the respective sheet names
result_sheets = pd.read_excel(result_path, sheet_name=None, engine='openpyxl')     	        #* Reload all updated sheets with avg values into a dictionary of DataFrames
overview_rows = []                                                                          #* List to store average values for the overview sheet
#header = df.columns.tolist()

# Loop through the dictionary of sheets
for sheet_name, df in result_sheets.items():
    if len(df) >= 422:                                             #* Ensure there are enough rows
        average_df = df.iloc[421].tolist()                         #* Extract previously created average values

        if not overview_rows:                                      #* This can also be kept outside the loop but was a bit slower (initialize header only once)
            header = ['Data File'] + df.columns.tolist()           #* Add sheet name to the header      

        average_df = [sheet_name] + average_df                     #* Add each sheet name to the average values
        overview_rows.append(average_df)                           #* Append the averages to the overview list
        print(f"Averaged values from {sheet_name} added to Overview sheet.")

    else:
        print(f"Sheet {sheet_name} does not have enough rows. Skipping...")

# Create the overview DataFrame
overview_df = pd.DataFrame(overview_rows, columns=header)

# Add the Overview sheet to the workbook
with pd.ExcelWriter(result_path, engine='openpyxl', mode='a') as writer:
    overview_df.to_excel(writer, sheet_name='Overview', index=False)

print(f"Completed Overview Sheet added to {result_path}")

Averaged values from 06_39_19 added to Overview sheet.
Averaged values from 07_00_59 added to Overview sheet.
Averaged values from 07_15_30 added to Overview sheet.
Averaged values from 07_30_17 added to Overview sheet.
Averaged values from 07_45_16 added to Overview sheet.
Averaged values from 08_00_16 added to Overview sheet.
Averaged values from 08_15_10 added to Overview sheet.
Averaged values from 08_29_57 added to Overview sheet.
Averaged values from 08_45_11 added to Overview sheet.
Averaged values from 09_00_20 added to Overview sheet.
Averaged values from 09_15_23 added to Overview sheet.
Averaged values from 09_30_11 added to Overview sheet.
Averaged values from 09_44_59 added to Overview sheet.
Averaged values from 10_00_05 added to Overview sheet.
Averaged values from 10_14_51 added to Overview sheet.
Averaged values from 10_30_08 added to Overview sheet.
Averaged values from 10_44_52 added to Overview sheet.
Averaged values from 10_59_40 added to Overview sheet.
Averaged v

In [16]:
from openpyxl import load_workbook

# Reorder sheets so overview is the first sheet and remove columns B-F 
wb = load_workbook(result_path)

if 'Overview' in wb.sheetnames:
    sheet_names = wb.sheetnames
    sheet_names.remove('Overview')
    wb._sheets = [wb['Overview']] + [wb[sheet] for sheet in sheet_names]

ws = wb['Overview']
for col in range (6, 1, -1):        #* Note: columns are 1-indexed, so range is 6 to 2 (B-F)
    col_letter = chr(64 + col)      #* Convert column number to letter
    ws.delete_cols(col)

wb.save(result_path)
print("Overview sheet moved to the first position and columns B-F removed.")


Overview sheet moved to the first position and columns B-F removed.


At this point we have to manually add the sample names corresponding to the data files (from the PTR-MS logbook)